# Day 3.2 — Context Budgets and Compaction

## Before you begin

### Learning outcomes

- Estimate the token cost of a growing message history.
- Compact an over-budget history and measure the reduction.
- Name exactly what the rule-based summary lost, and why that motivates real memory.

Architecture reference: [Day 3 diagrams D09](../../diagrams/source/day_03.md).

### Expected observation

Before compaction the history is over budget; after compaction it fits, with a visible summary message at the front. The oldest fact (a deadline) is gone - that loss is the point of the lesson.

## Concept briefing

## Context budgets and compaction

History grows on every turn. It costs tokens, adds latency, and eventually exceeds the
model's context window, so the application has to decide what to drop.

Compaction keeps recent turns verbatim and replaces older turns with a shorter summary.
Two kinds of summary appear in practice:

- a **rule-based** summary, written by our own Python code. It is cheap, deterministic and
  free, but it has no idea which words mattered: it simply keeps the first few words of
  each older turn and drops the oldest turns when the summary budget runs out;
- a **model** summary, written by a second model call. It reads better and can compress
  several turns into one sentence, but it costs a call, is not deterministic, and can
  quietly omit, merge or reword a fact.

Both are lossy. There is no compression that preserves every future-relevant detail
without knowing the future questions. That is the argument for persistent memory: a fact
you must not lose does not belong in a summary, it belongs in a store you control.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/safe_task_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

## Step 1 — Estimating tokens without a tokenizer

Providers bill per token. A rough classroom rule is *four characters ≈ one token*, which is
what `estimate_tokens` implements. It is approximate on purpose: we want a number we can see,
not an exact bill.

In [ ]:
from safe_task_agent import Message, estimate_tokens, total_tokens, compact_history

sample = "The project deadline is 14 March and it cannot move."
print("Text       :", sample)
print("Characters :", len(sample))
print("Estimated tokens:", estimate_tokens(sample), "  (characters / 4, rounded up)")

## Step 2 — Build a conversation that has run for a while

Ten user/assistant pairs. Notice that the *first* exchange contains the most valuable fact in
the whole conversation: the deadline.

In [ ]:
pairs = [
    ("My final year project is called Aurora and the hard deadline is 14 March; it cannot move.",
     "Recorded: project Aurora, hard deadline 14 March."),
    ("Every measurement in the report must be written in millimetres, never in inches.",
     "Understood, all measurements will use millimetres."),
    ("The lab session moved from Tuesday morning to Thursday afternoon this week.",
     "Noted, the lab session is now on Thursday afternoon."),
    ("My mentor prefers short emails, at most five sentences, with a clear subject line.",
     "Understood, emails to your mentor will stay under five sentences."),
    ("Please keep every progress summary under one hundred words.",
     "Agreed, progress summaries will stay under one hundred words."),
    ("The sensor board we ordered arrives next Monday, so testing starts after that.",
     "Noted, testing starts after the sensor board arrives next Monday."),
    ("Team meetings should never be scheduled before ten in the morning.",
     "Understood, no meetings before ten."),
    ("The report template asks for a one page abstract at the front.",
     "Noted, a one page abstract goes at the front."),
    ("Please use the university logo only on the cover page of the report.",
     "Understood, the logo stays on the cover page."),
    ("Remind me which measurement unit we agreed on for the report.",
     "Millimetres, as you asked earlier."),
]

history = []
for user_text, assistant_text in pairs:
    history.append(Message("user", user_text))
    history.append(Message("assistant", assistant_text))

print("Messages         :", len(history))
print("Estimated tokens :", total_tokens(history))
print("Oldest message   :", history[0].content)

## Step 3 — Give the context an artificial budget

Real context windows are large, so we shrink the budget to make the problem visible in class.
Everything you learn here applies unchanged when the number is 128,000 instead of 200.

In [ ]:
BUDGET = 200                       # artificially small so the lesson fits on screen

before_tokens = total_tokens(history)
print("Budget           :", BUDGET, "tokens")
print("History costs    :", before_tokens, "tokens")
print("Over budget?     :", before_tokens > BUDGET, f"(by {before_tokens - BUDGET} tokens)")

## Step 4 — Compact, and measure the reduction

`compact_history` keeps the most recent turns word-for-word and replaces the older ones with a
single summary message. The summary gets its own budget (`BUDGET // 3`), so it can never grow
back into a copy of the conversation.

In [ ]:
compacted = compact_history(history, budget=BUDGET)
after_tokens = total_tokens(compacted)

print("BEFORE:", len(history), "messages,", before_tokens, "tokens")
print("AFTER :", len(compacted), "messages,", after_tokens, "tokens")
print("Saved :", before_tokens - after_tokens, "tokens",
      f"({100 * (before_tokens - after_tokens) // before_tokens}% smaller)")
print("Fits inside the budget?", after_tokens <= BUDGET)

## Step 5 — Read the summary itself

Compaction must never be invisible. Print the summary message and the recent turns that were
kept verbatim.

In [ ]:
print("--- the summary message that replaced the old turns ---")
print(compacted[0].content)

print("\n--- kept verbatim (most recent turns) ---")
for message in compacted[1:]:
    print(f"{message.role:>9}: {message.content}")

## Step 6 — Name what was lost

The summary is produced by **our own Python code**, not by a model. It keeps the first few words
of each dropped turn and discards the oldest turns first when its budget runs out. So the loss is
predictable and free, but blind: nothing in the code knows that "14 March" mattered more than
"Understood".

In [ ]:
deadline_before = any("14 March" in m.content for m in history)
deadline_after = any("14 March" in m.content for m in compacted)

print("Deadline present before compaction:", deadline_before)
print("Deadline present after compaction :", deadline_after)
print("\nThe summary is RULE-BASED: no model was called, so nothing was invented,")
print("but nothing was understood either - the oldest turn was simply dropped.")
print("A MODEL summary would reword the old turns and might well keep '14 March',")
print("but it costs an extra call, is not deterministic, and can also quietly drop")
print("or alter a fact. Either way the summary is lossy.")
print("\nConclusion: a fact you must not lose does not belong in a summary.")
print("It belongs in an explicit memory store - that is Day 3.3.")

### Try it yourself

Predict the direction first: if you give compaction a **larger** budget, does the summary keep
more facts or fewer? Run the loop below to check.

In [ ]:
# --- Worked solution ---
# Run the same history through three budgets and print what each one produces.
for budget in (120, 200, 280):
    result = compact_history(history, budget=budget)
    # Every kept fact is one "- role: text" line; the "(N older turns dropped)" line is not a fact.
    facts = [line for line in result[0].content.splitlines()
             if line.startswith("- ") and "dropped)" not in line]
    print(f"budget={budget:>3}  ->  {total_tokens(result):>3} tokens, "
          f"{len(result):>2} messages, {len(facts)} facts kept in the summary")

# Larger budget -> more recent turns kept verbatim AND a bigger summary allowance,
# so more of the old facts survive. The trade is money and latency on every call.

### Checkpoint

**1. Why does compaction shrink the history at all, instead of just moving the old text into a summary message?**

<details><summary>Show answer</summary>

Because the summary has a budget of its own (`budget // 3`). Facts are added newest first and the oldest are dropped when that budget is full, so the summary is always much smaller than the turns it replaced. The old version of this function pasted the old turns into one long line and saved nothing.

</details>

**2. Our summary is written by Python. What changes if a model writes it instead?**

<details><summary>Show answer</summary>

It reads better and can genuinely compress several turns into one sentence, so an important old fact is more likely to survive. But it costs another model call, the output varies between runs, and it can reword or silently drop a detail. Lossy either way - which is why durable facts go into memory, not into a summary.

</details>

### Recap

- **Limitation we saw:** A conversation grew past its context budget, and compacting it destroyed the deadline stated in the very first turn.
- **Layer we added:** A budgeted, printable compaction step between the history and the model call.
- **Evidence it worked:** 315 tokens in 20 messages became a summary plus recent turns inside the 200-token budget, with the lost fact named explicitly.